# Arithmetic Coding — Lossless Compression

This notebook is part of the Digital Image Processing project.

## Part C — Arithmetic Coding


# Arithmetic Coding – Lossless Compression

Arithmetic coding represents a complete message by a value inside a progressively narrowed interval. The decoder uses the same symbol probabilities and the known message length to recover the original sequence.

This notebook uses Python's `Decimal` arithmetic to reduce floating-point precision errors for moderate-length examples.

In [ ]:
from collections import Counter
from decimal import Decimal, getcontext

def build_probability_ranges(text):
    if not text:
        return {}, {}
    frequencies = Counter(text)
    total = Decimal(len(text))
    ranges = {}
    cumulative = Decimal(0)
    for symbol in sorted(frequencies):
        probability = Decimal(frequencies[symbol]) / total
        next_cumulative = cumulative + probability
        ranges[symbol] = (cumulative, next_cumulative)
        cumulative = next_cumulative
    return frequencies, ranges

def arithmetic_encode(text):
    if not text:
        return Decimal("0.5"), {}, {}

    getcontext().prec = max(100, 2 * len(text) + 30)
    frequencies, ranges = build_probability_ranges(text)
    low, high = Decimal(0), Decimal(1)

    for symbol in text:
        width = high - low
        symbol_low, symbol_high = ranges[symbol]
        new_high = low + width * symbol_high
        new_low = low + width * symbol_low
        low, high = new_low, new_high

    return (low + high) / 2, frequencies, ranges

def arithmetic_decode(encoded_value, frequencies, ranges, message_length):
    if message_length == 0:
        return ""
    value = Decimal(encoded_value)
    low, high = Decimal(0), Decimal(1)
    symbols = []

    for _ in range(message_length):
        width = high - low
        if width <= 0:
            raise ArithmeticError("Interval precision exhausted; use a shorter message.")
        scaled_value = (value - low) / width

        selected = None
        for symbol, (symbol_low, symbol_high) in ranges.items():
            if symbol_low <= scaled_value < symbol_high:
                selected = symbol
                new_low = low + width * symbol_low
                new_high = low + width * symbol_high
                break

        if selected is None:
            # Decimal rounding at the upper boundary can occur in extreme cases.
            symbol, (symbol_low, symbol_high) = next(reversed(list(ranges.items())))
            selected = symbol
            new_low = low + width * symbol_low
            new_high = low + width * symbol_high

        symbols.append(selected)
        low, high = new_low, new_high

    return "".join(symbols)

text = input("Enter text to encode: ")
encoded_value, frequencies, ranges = arithmetic_encode(text)
decoded_text = arithmetic_decode(encoded_value, frequencies, ranges, len(text))

print("Original text:", repr(text))
print("\nCharacter frequencies and probability intervals:")
for symbol in sorted(frequencies):
    print(repr(symbol), "frequency =", frequencies[symbol],
          "interval =", ranges[symbol])
print("\nArithmetic encoded value:", encoded_value)
print("Decoded text:", repr(decoded_text))
print("Decoding successful:", decoded_text == text)


## Note on arithmetic coding

The encoded decimal value is a mathematical representation of the final interval, not a complete practical compressed file format. A production codec also stores the probability model/message length and uses finite-precision bit output with renormalization. The notebook demonstrates interval encoding and decoding for educational examples.